In [20]:
import pandas as pd
import glob
import re
from textblob import TextBlob
from collections import Counter
import fasttext
import numpy as np
from multiprocessing import Pool
import json


In [ ]:
# Combining Data files into 1 file
# data_path = '../raw_data/reviews/'
# gz_files = sorted(glob.glob(data_path + 'reviews.csv*.gz'))
# print(f"Found {len(gz_files)} files")
# dfs = [pd.read_csv(f, compression='gzip') for f in gz_files]
# reviews = pd.concat(dfs, ignore_index=True)
# print(f"Combined: {reviews.shape[0]} rows, {reviews.shape[1]} columns")
# # Drop reviewers name
# reviews.drop(['reviewer_name'], axis=1, inplace=True)
# reviews.to_csv(r'../final_files/reviews.csv', index=False)

Found 11 files
Combined: 10824788 rows, 6 columns


# Cleaning Reviews

#### Importing combined data

In [2]:
reviews = pd.read_csv(r'../final_files/reviews.csv')

In [4]:
# ── Reviewer frequency analysis ──
print("=== Reviewer Frequency ===")
print(reviews['reviewer_id'].value_counts())
 
x = reviews["reviewer_id"].value_counts().reset_index()
pct = x[x['count'] > 1].shape[0] / x.shape[0] * 100
print(f"\n{pct:.2f}% of reviewers have multiple reviews")
 
print("\nQuantiles:")
print(x['count'].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
 
# ── Bin reviewers by review count ──
bins = [0, 5, 10, 15, 20, 25, 35, 45, 55, 65, 75, 85, 95, 100, x['count'].max()]
labels = ['1-5', '6-10', '11-15', '16-20', '21-25', '26-35', '36-45',
          '46-55', '56-65', '66-75', '76-85', '86-95', '96-100', '100+']
 
x['bin'] = pd.cut(x['count'], bins=bins, labels=labels, right=True)
freq = x['bin'].value_counts().sort_index().reset_index()
freq.columns = ['Review Count', 'Num Reviewers']
freq['Percentage'] = (freq['Num Reviewers'] / freq['Num Reviewers'].sum() * 100).round(2)
print("\nReview count distribution:")
print(freq)

=== Reviewer Frequency ===
reviewer_id
63643121     704
356757832    621
564445127    549
392403224    502
356658445    468
            ... 
549670347      1
556353920      1
580453220      1
424994311      1
393361825      1
Name: count, Length: 1003620, dtype: int64

97.80% of reviewers have multiple reviews

Quantiles:
0.25    10.0
0.50    11.0
0.75    11.0
0.90    11.0
0.95    22.0
0.99    33.0
Name: count, dtype: float64

Review count distribution:
   Review Count  Num Reviewers  Percentage
0           1-5         134920       13.44
1          6-10         140840       14.03
2         11-15         647676       64.53
3         16-20          14879        1.48
4         21-25          44583        4.44
5         26-35          12442        1.24
6         36-45           4168        0.42
7         46-55           1799        0.18
8         56-65            599        0.06
9         66-75            591        0.06
10        76-85            367        0.04
11        86-95           

In [5]:
# -- Clean text for sentiment analysis --
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)       # remove URLs
    text = re.sub(r'<.*?>', '', text)                    # remove HTML tags
    text = re.sub(r'[^a-z\s\']', ' ', text)             # keep apostrophes
    text = re.sub(r'\s+', ' ', text).strip()            # collapse spaces
    return text

print("\nCleaning text...")
reviews['clean_comments'] = reviews['comments'].apply(clean_text)



Cleaning text...


In [8]:
# -- Language detection --

model = fasttext.load_model('../model_objects/lid.176.bin')

def detect_language(text):
    text = str(text).strip()
    if text == "" or text == "nan" or text == "None":
        return "unknown"
    try:
        prediction = model.predict(text.replace('\n', ' '), k=1)
        lang = prediction[0][0].replace('__label__', '')
        return lang
    except:
        return "unknown"

In [10]:
# Quick test
for comment in reviews['comments'].head(10):
    print(comment, "-->", detect_language(comment))

Great host  --> en
Nice room for the price. Great neighborhood. John was very accommodating. Bottles of water in the room were a nice touch and very much appreciated. --> en
Very nice apt.  New remodeled. --> en
Great place to stay for a while. John is a great host and a great man. The house is very homy. You'll feel right at home.  --> en
   . --> fr
I really enjoyed my time here in deep south Brooklyn and John was an excellent host - very attentive to any concerns that I had and a very welcome presence when he's around. <br/>The room itself is a great space with a super-comfy bed and lovely traditional ceiling fan. The street is pleasant, tree-lined and quiet and the townhouse is located a short, convenient walk from the F train, which has been perfect for getting to my office here.<br/>It's easy to see why John's a Super Host and I look forward to staying with him again! --> en
John was a great host. He was very responsive before and during our visit. He really wanted to make sure w

In [11]:
reviews['language'] = reviews['comments'].apply(detect_language)
print(reviews['language'].value_counts().head(10))

# Filter to English and save
english_reviews = reviews[reviews['language'] == 'en'].copy()
print(f"\nTotal: {len(reviews)}, English: {len(english_reviews)}")

english_reviews.to_csv(r'../final_files/reviews_with_language.csv', index=False)
print("Saved!")

language
en    9540439
es     394881
fr     374064
de     162485
it      73947
pt      67064
zh      44643
nl      31731
ko      31595
ja      20587
Name: count, dtype: int64

Total: 10824788, English: 9540439
Saved!


# Performing Sentiment Analysis

##### Importing Language tagged Dataset

In [21]:
df = pd.read_csv(r'../final_files/reviews_with_language.csv')

In [22]:
def get_sentiment(text):
    text = str(text).strip()
    if text == "" or text == "nan":
        return 0.0
    try:
        return TextBlob(text).sentiment.polarity
    except:
        return 0.0



In [23]:
## Extracting Sentiment Score
print("Running sentiment analysis with 32 cores...")
with Pool(32) as pool:
    df['sentiment_score'] = pool.map(get_sentiment, df['clean_comments'])
print("Done!")

df['sentiment_label'] = df['sentiment_score'].apply(
    lambda x: 'positive' if x > 0 else ('negative' if x < 0 else 'neutral')
)

print(df['sentiment_label'].value_counts())
print(f"\nMean sentiment score: {df['sentiment_score'].mean():.4f}")


Running sentiment analysis with 32 cores...
Done!
sentiment_label
positive    9275452
neutral      135527
negative     129460
Name: count, dtype: int64

Mean sentiment score: 0.4122


In [ ]:

# The paper removes vague/uninformative reviews
vague_patterns = ['good', 'very good', 'nice', 'ok', 'okay', 'fine', 'great',
                  'not bad', 'not so good', 'excellent', 'wonderful']

df['is_vague'] = df['clean_comments'].apply(lambda x: str(x).strip() in vague_patterns)
print(f"Vague reviews: {df['is_vague'].sum()} ({df['is_vague'].mean()*100:.2f}%)")

df_filtered = df[~df['is_vague']].copy()
print(f"Remaining reviews: {len(df_filtered)}")

Vague reviews: 39442 (0.41%)
Remaining reviews: 9500997


In [25]:
# The paper segments homestays into 3 categories based on
# sentiment polarity distribution per listing

# Calculate positive sentiment ratio per listing
listing_seg = df_filtered.groupby('listing_id').agg(
    mean_sentiment=('sentiment_score', 'mean'),
    median_sentiment=('sentiment_score', 'median'),
    num_reviews=('sentiment_score', 'count'),
    positive_ratio=('sentiment_label', lambda x: (x == 'positive').mean()),
    negative_ratio=('sentiment_label', lambda x: (x == 'negative').mean()),
    neutral_ratio=('sentiment_label', lambda x: (x == 'neutral').mean())
).reset_index()

# Segment based on the paper's approach:
# Satisfactory: predominantly positive (positive_ratio >= 0.9)
# Moderate: mixed sentiment (0.5 <= positive_ratio < 0.9)
# Dissatisfactory: predominantly negative (positive_ratio < 0.5)

def segment_listing(row):
    if row['positive_ratio'] >= 0.9:
        return 'satisfactory'
    elif row['positive_ratio'] >= 0.5:
        return 'moderate'
    else:
        return 'dissatisfactory'

listing_seg['segment'] = listing_seg.apply(segment_listing, axis=1)

print("Homestay Segmentation:")
print(listing_seg['segment'].value_counts())
print(f"\n{listing_seg['segment'].value_counts(normalize=True).mul(100).round(2)}")

Homestay Segmentation:
segment
satisfactory       26435
moderate            2001
dissatisfactory      354
Name: count, dtype: int64

segment
satisfactory       91.82
moderate            6.95
dissatisfactory     1.23
Name: proportion, dtype: float64


In [26]:
listing_seg.head()

,listing_id,mean_sentiment,median_sentiment,num_reviews,positive_ratio,negative_ratio,neutral_ratio,segment
0,2539,0.477870,0.492727,72,1.000000,0.000000,0.000000,satisfactory
1,2595,0.351413,0.358462,481,0.977131,0.000000,0.022869,satisfactory
2,5136,0.376293,0.414806,12,1.000000,0.000000,0.000000,satisfactory
3,6848,0.384601,0.366270,1946,0.977390,0.016958,0.005653,satisfactory
4,6872,0.415521,0.537407,15,1.000000,0.000000,0.000000,satisfactory


In [ ]:
## Save listing Sentiment Segmentation - Sentiment Scores Aggregated over listings
listing_seg.to_csv(r'../final_files/listing_sentiment_segments.csv', index=False)

print("Saved listing_sentiment_segments.csv")

Saved listing_sentiment_segments.csv


# Deriving the other 7 Determinants

#### Importing and Combining Listings Data

In [ ]:
# # Combining Listing files into 1 file
# data_path = '../raw_data/listing/'
# gz_files = sorted(glob.glob(data_path + 'listings*.csv.gz'))
# print(f"Found {len(gz_files)} files")
# dfs = [pd.read_csv(f, compression='gzip', low_memory=False) for f in gz_files]
# listings = pd.concat(dfs, ignore_index=True)
# print(f"Combined: {listings.shape[0]} rows, {listings.shape[1]} columns")

# # Exporting the file
# listings.to_csv(r'../final_files/listings.csv', index=False)
# print("Exported!!")

Found 11 files
Combined: 400865 rows, 85 columns
Exported!!


In [ ]:
# Importing Combined Listings Data
listings = pd.read_csv(r'../final_files/listings.csv')

In [ ]:
# -- Extract 7 determinants from listings --
# Count amenities
def count_amenities(x):
    try:
        return len(json.loads(x))
    except:
        return 0

listings['amenities_count'] = listings['amenities'].apply(count_amenities)

# Clean percentage columns
listings['host_response_rate_clean'] = listings['host_response_rate'].str.replace('%', '').astype(float) / 100
listings['host_acceptance_rate_clean'] = listings['host_acceptance_rate'].str.replace('%', '').astype(float) / 100

# Encode room_type
room_type_map = {'Entire home/apt': 3, 'Private room': 2, 'Shared room': 1, 'Hotel room': 2}
listings['room_type_encoded'] = listings['room_type'].map(room_type_map)

# Encode instant_bookable
listings['instant_bookable_encoded'] = listings['instant_bookable'].map({'t': 1, 'f': 0})

# Select determinants
determinants = listings[['id',
    'review_scores_cleanliness',
    'review_scores_communication',
    'review_scores_location',
    'amenities_count',
    'room_type_encoded',
    'instant_bookable_encoded',
    'host_response_rate_clean',
]].copy()

determinants.columns = ['listing_id', 'cleaning_score', 'communication_score',
                         'location_score', 'amenities', 'room_type',
                         'instant_bookable', 'response_rate']

# Aggregate across months per listing
determinants = determinants.groupby('listing_id').agg(
    cleaning_score=('cleaning_score', 'mean'),
    communication_score=('communication_score', 'mean'),
    location_score=('location_score', 'mean'),
    amenities=('amenities', 'max'),          # take latest/max count
    room_type=('room_type', 'median'),        # most common encoding
    instant_bookable=('instant_bookable', 'median'),
    response_rate=('response_rate', 'mean')
).reset_index()

print("Missing values:")
print(determinants.isnull().sum())
print(f"\nUnique listings: {len(determinants)}")
print(determinants.describe())

Missing values:
listing_id                 0
cleaning_score         15350
communication_score    15355
location_score         15366
amenities                  0
room_type                  0
instant_bookable        1371
response_rate          15090
dtype: int64

Unique listings: 44578
         listing_id  cleaning_score  communication_score  location_score  \
count  4.457800e+04    29228.000000         29223.000000    29212.000000   
mean   5.635554e+17        4.649644             4.809435        4.738594   
std    5.893347e+17        0.536884             0.459906        0.423964   
min    2.539000e+03        0.000000             0.000000        0.000000   
25%    2.590524e+07        4.540000             4.810909        4.657121   
50%    5.882688e+17        4.820000             4.960000        4.860000   
75%    1.111632e+18        5.000000             5.000000        5.000000   
max    1.619352e+18        5.000000             5.000000        5.000000   

          amenities     room_t

In [ ]:
final_data = determinants.merge(listing_seg, on='listing_id', how='inner')
print(f"Final dataset: {len(final_data)} listings, {final_data.shape[1]} columns")
print(final_data.head())
# This is the final dataset
final_data.to_csv(r'../final_files/listings_with_determinants.csv', index=False)
print("Saved!")

Final dataset: 28790 listings, 15 columns
   listing_id  cleaning_score  communication_score  location_score  amenities  \
0        2539            5.00                  5.0            4.75         56   
1        2595            4.63                  4.8            4.81         32   
2        5136            4.50                  5.0            4.75         44   
3        6848            4.85                  4.8            4.69         26   
4        6872            5.00                  5.0            5.00         18   

   room_type  instant_bookable  response_rate  mean_sentiment  \
0        2.0               0.0       1.000000        0.477870   
1        3.0               0.0       0.705556        0.351413   
2        3.0               1.0       0.000000        0.376293   
3        3.0               0.0       1.000000        0.384601   
4        2.0               0.0       0.918889        0.415521   

   median_sentiment  num_reviews  positive_ratio  negative_ratio  \
0          0